# 4a Handoff github

In [4]:
import os
from enum import Enum
from pydantic import BaseModel
from rich import print
from dotenv import load_dotenv
import asyncio

In [5]:
load_dotenv("../.env/bailian", override=True)

True

In [6]:
model_id = os.environ["OPENAI_RESPONSES_MODEL_ID"]
model_id

'qwen3.5-plus'

In [7]:
from semantic_kernel.functions import kernel_function
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.agents import ChatCompletionAgent, HandoffOrchestration
from semantic_kernel.contents import ChatMessageContent, AuthorRole
from semantic_kernel.agents.runtime import InProcessRuntime

In [8]:
class GitHubLabels(Enum):
    """Enum representing GitHub labels."""

    PYTHON = "python"
    DOTNET = ".NET"
    BUG = "bug"
    ENHANCEMENT = "enhancement"
    QUESTION = "question"
    VECTORSTORE = "vectorstore"
    AGENT = "agent"


class GithubIssue(BaseModel):
    """Model representing a GitHub issue."""

    id: str
    title: str
    body: str
    labels: list[str] = []


class Plan(BaseModel):
    """Model representing a plan for resolving a GitHub issue."""

    tasks: list[str]

In [9]:
class GithubPlugin:
    """Plugin for GitHub related operations."""

    @kernel_function
    async def add_labels(self, issue_id: str, labels: list[GitHubLabels]) -> None:
        """Add labels to a GitHub issue."""
        await asyncio.sleep(1)  # Simulate network delay
        print(f"Adding labels {labels} to issue {issue_id}")

    @kernel_function(description="Create a plan to resolve the issue.")
    async def create_plan(self, issue_id: str, plan: Plan) -> None:
        """Create tasks for a GitHub issue."""
        await asyncio.sleep(1)  # Simulate network delay
        print(f"Creating plan for issue {issue_id} with tasks:\n{plan.model_dump_json(indent=2)}")

In [10]:
triage_agent = ChatCompletionAgent(
    name="TriageAgent",
    description="An agent that triages GitHub issues",
    instructions="Given a GitHub issue, triage it.",
    service=OpenAIChatCompletion(),
)
python_agent = ChatCompletionAgent(
    name="PythonAgent",
    description="An agent that handles Python related issues",
    instructions="You are an agent that handles Python related GitHub issues.",
    service=OpenAIChatCompletion(),
    plugins=[GithubPlugin()],
)
dotnet_agent = ChatCompletionAgent(
    name="DotNetAgent",
    description="An agent that handles .NET related issues",
    instructions="You are an agent that handles .NET related GitHub issues.",
    service=OpenAIChatCompletion(),
    plugins=[GithubPlugin()],
)

In [ ]:
handoffs = {
    triage_agent.name: {
        python_agent.name: "Transfer to this agent if the issue is Python related",
        dotnet_agent.name: "Transfer to this agent if the issue is .NET related",
    },
}

In [12]:
agents = [triage_agent, python_agent, dotnet_agent]

In [13]:
def custom_input_transform(input_message: GithubIssue) -> ChatMessageContent:
    return ChatMessageContent(role=AuthorRole.USER, content=input_message.model_dump_json())

In [ ]:
hoor = handoff_orchestration = HandoffOrchestration(
    members=agents,
    handoffs=handoffs,
    # input_transform必须返回ChatMessageContent
    input_transform=custom_input_transform,
)

In [15]:
GithubIssueSample = GithubIssue(
    id="12345",
    title=(
        "Bug: SQLite Error 1: 'ambiguous column name:' when including VectorStoreRecordKey in "
        "VectorSearchOptions.Filter"
    ),
    body=(
        "Describe the bug"
        "When using column names marked as [VectorStoreRecordData(IsFilterable = true)] in "
        "VectorSearchOptions.Filter, the query runs correctly."
        "However, using the column name marked as [VectorStoreRecordKey] in VectorSearchOptions.Filter, the query "
        "throws exception 'SQLite Error 1: ambiguous column name: StartUTC"
        ""
        "To Reproduce"
        "Add a filter for the column marked [VectorStoreRecordKey]. Since that same column exists in both the "
        "vec_TestTable and TestTable, the data for both columns cannot be returned."
        ""
        "Expected behavior"
        "The query should explicitly list the vec_TestTable column names to retrieve and should omit the "
        "[VectorStoreRecordKey] column since it will be included in the primary TestTable columns."
        ""
        "Platform"
        ""
        "Microsoft.SemanticKernel.Connectors.Sqlite v1.46.0-preview"
        "Additional context"
        "Normal DBContext logging shows only normal context queries. Queries run by VectorizedSearchAsync() don't "
        "appear in those logs and I could not find a way to enable logging in semantic search so that I could "
        "actually see the exact query that is failing. It would have been very useful to see the failing semantic "
        "query."
    ),
    labels=[],
)

In [ ]:
runtime = InProcessRuntime()
runtime.start()

hoor_result = await hoor.invoke(
    task=GithubIssueSample,
    runtime=runtime,
)

value = await hoor_result.get()

Adding labels [<GitHubLabels.DOTNET: '.NET'>, <GitHubLabels.BUG: 'bug'>, <GitHubLabels.VECTORSTORE: 'vectorstore'>]
to issue 12345

Creating plan for issue 12345 with tasks:
{
  "tasks": [
    "Investigate the query generation logic in Microsoft.SemanticKernel.Connectors.Sqlite to identify how column 
names are resolved during VectorSearchOptions.Filter execution.",
    "Determine why the [VectorStoreRecordKey] column causes an 'ambiguous column name' error by analyzing the 
generated SQL query, particularly focusing on joins between vec_TestTable and TestTable.",
    "Modify the query generation to explicitly qualify column names with their respective table aliases, ensuring 
the [VectorStoreRecordKey] column is not duplicated or ambiguously referenced.",
    "Ensure the primary TestTable's [VectorStoreRecordKey] column is used appropriately while omitting it from 
vec_TestTable's selected columns.",
    "Add logging capabilities to capture and surface the exact semantic query being executed for debugging 
purposes."
  ]
}

In [18]:
print(value.content)

Task is completed with summary: The issue involves a SQLite 'ambiguous column name' error when using 
[VectorStoreRecordKey] in VectorSearchOptions.Filter due to unqualified column references in generated queries. A 
plan has been created to investigate and fix the query generation logic, ensure proper column qualification, and 
add logging for transparency.

In [ ]:
await runtime.stop_when_idle()